## Notebook Name: Conservation Areas Transform
**Medallion Layer: Silver**  

**Purpose:** Transform the Raw Data of the Conservation Areas that shows the point locations into transformed silver layer table

**Author:** Matthew Kristanto  

**Date Created:** 4/03/26  

**Last Modified:** 5/03/26  

**Source Table:** [bronze_conservation_areas_ingest](https://adb-7405618007730451.11.azuredatabricks.net/editor/notebooks/1223121134226800?o=7405618007730451)

**Notes:**
- Checks that each Attribute belongs to the correct data type
- Convert Latitude and Longitude to Double Types
- Does handling of NULL or Missing Values for each of the attributes
- NULL Values in each attributes per Record would be replaced by "Unknown" execpt for Latitude and Longitude being Double Types, so they still keep NULL Value
- Checks for any Records that have been duplicated  
- It is assumed that a duplication has occured when the values for Name, County, Municipality, Region and Areas of Interest are the same
- Split Georeference into Georeference_Longitude and Georeference_Latitude
- Check the Range for Latitude and Longitude is between -90 and 90, as well as -180 and 180 respectively.
- A Column is created that checks if the Latitude, Longitude, Georeference_Longitude and the Georeference_Latitude is Valid or not


In [0]:
### Retrieve the Bronze Table

df_bronze = spark.read.table("bronze.bronze_bird_conservation_areas")

display(df_bronze.limit(5))

In [0]:
### Check the Data Types of the Attributes

df_bronze.printSchema()

In [0]:
### Convert Latitude and Longitude to Double Types
df_silver = df_bronze.withColumn("Latitude", df_bronze["Latitude"].cast("double"))
df_silver = df_silver.withColumn("Longitude", df_bronze["Longitude"].cast("double"))

df_silver.printSchema()

In [0]:
from pyspark.sql.functions import count, col

### Check for Duplication of Records
duplicate_check = (
  df_silver.groupBy(
    "Name",
    "County",
    "Municipality",
    "Region",
    "Areas_of_Interest"
  )
  ### Keep only the duplicates to display
  .agg(count("*").alias("count"))
  .filter(col("count") > 1)
)

### Use Inner Join to get the Records who have the same Combinations
duplicates = df_silver.join(
  duplicate_check,
    on=["Name", "County", "Municipality", "Region", "Areas_of_Interest"],
    how="inner"
)

display(duplicates)


In [0]:
from pyspark.sql.functions import sum, when, col

def get_null_counts(df):
    ### Check for Null Values
    null_counts_expr = []

    for c in df_bronze.columns:
        ## Add count 1, if there is a Null Value
        null_count = sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        null_counts_expr.append(null_count)
        
    display(df.select(null_counts_expr))

get_null_counts(df_silver)

In [0]:
### Fill the Missing Values with Placeholders
df_silver = df_silver.fillna({
    "Name": "Unknown",
    "County": "Unknown",
    "Municipality": "Unknown",
    "Region": "Unknown",
    "Areas_of_Interest": "Unknown",
    "Key_BCA_Criteria": "Unknown",
    "Critical_Habitat_Types": "Unknown",
    "Georeference": "Unknown"
})

In [0]:
### Check missing values
get_null_counts(df_silver)

In [0]:
### Split Georeference into Georeference Latitude and Georeference Longitude
from pyspark.sql.functions import regexp_replace, split

## Have an Intermediate Column that removes the Brackets and spaces at end of string
df_silver = df_silver.withColumn("Georeference_Clean", regexp_replace(col("Georeference"), "[()]|,\\s*$", ""))

## Seperates based on Comma
df_silver = df_silver.withColumn("Georeference_Latitude", split(col("Georeference_Clean"), ",")[0].cast("double"))
df_silver = df_silver.withColumn("Georeference_Longitude", split(col("Georeference_Clean"), ",")[1].cast("double"))

## Drop the Old Geoference Column and the Intermediate Column
df_silver = df_silver.drop("Georeference_Clean")
df_silver = df_silver.drop("Georeference")

display(df_silver.limit(5))


In [0]:
### Ensure that the range of Longitude and Latitude is valid
invalid_lat_long = df_silver.filter((col("Latitude") < -90) | (col("Latitude") > 90) | (col("Longitude") < -180) | (col("Longitude") > 180))

display(invalid_lat_long)

In [0]:
### Ensure that the range of Georeference Longitude and Georeference Latitude is valid

invalid_georeference_lat_long = df_silver.filter((col("Latitude") < - 90) | (col("Latitude") > 90) | (col("Longitude") < -180) | (col("Longitude") > 180))

display(invalid_georeference_lat_long)

In [0]:
### Create another Column that informs if valid

df_silver = df_silver.withColumn(
    "Coordinates_Valid",
    when(
        (col("Latitude").between(-90, 90)) &
        (col("Longitude").between(-180, 180)),
        True
    ).otherwise(False)
)

In [0]:
df_silver = df_silver.withColumn(
    "Georeference_Coordinates_Valid",
    when(
        (col("Georeference_Latitude").between(-90, 90)) &
        (col("Georeference_Longitude").between(-180, 180)),
        True
    ).otherwise(False)
)

In [0]:
### Create Silver Schema if not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
# Now add the Delta Table
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.silver_bird_conservation_areas_cleaned"))

In [0]:
%sql
SELECT * FROM silver.silver_bird_conservation_areas_cleaned